# Tool Calling

In [7]:
import dotenv
from agents import Agent, ModelSettings, Runner, function_tool, trace

dotenv.load_dotenv()

True

Create a calculator function that the Math Tutor can use as a tool:

In [8]:
def calculate_math(operation: str, first_number: float, second_number: float) -> str:
    """Calculate a basic arithmetic operation on two numbers.

    Args:
        operation: One of add, subtract, multiply, or divide.
        first_number: The first number in the calculation.
        second_number: The second number in the calculation.
    """
    operation = operation.strip().lower()
    if operation == "add":
        result = first_number + second_number
        symbol = "+"
    elif operation == "subtract":
        result = first_number - second_number
        symbol = "-"
    elif operation == "multiply":
        result = first_number * second_number
        symbol = "*"
    elif operation == "divide":
        if second_number == 0:
            raise ValueError("Cannot divide by zero.")
        result = first_number / second_number
        symbol = "/"
    else:
        raise ValueError("operation must be add, subtract, multiply, or divide.")

    return f"{first_number:g} {symbol} {second_number:g} = {result:g}"

Add an EVS concept lookup function for the Environmental Studies tutor:

In [9]:
def lookup_evs_concept(concept: str) -> str:
    """Look up a basic explanation of an environmental science concept."""
    concept_data = {
        "photosynthesis": (
            "Plants use sunlight, water, and carbon dioxide to make glucose and release oxygen."
        ),
        "ozone depletion": (
            "The thinning of the ozone layer, mainly caused by chemicals such as CFCs, "
            "allows more harmful ultraviolet radiation to reach Earth."
        ),
        "global warming": (
            "The long-term rise in Earth's average temperature, primarily caused by "
            "greenhouse gases released through human activities."
        ),
        "greenhouse effect": (
            "Atmospheric gases trap some of Earth's heat. This natural process keeps "
            "Earth warm, but extra greenhouse gases strengthen it."
        ),
    }

    concept_key = concept.strip().lower()
    if concept_key in concept_data:
        return f"{concept_key.title()}: {concept_data[concept_key]}"

    available_concepts = ", ".join(concept_data)
    return f"I don't have information about '{concept}'. Try one of these: {available_concepts}."


evs_lookup_tool = function_tool(lookup_evs_concept)

Test the regular Python functions before registering them as agent tools:

In [10]:
calculate_math("multiply", 7, 8)

'7 * 8 = 56'

In [11]:
lookup_evs_concept("photosynthesis")

'Photosynthesis: Plants use sunlight, water, and carbon dioxide to make glucose and release oxygen.'

In [12]:
math_tool = function_tool(calculate_math)

math_tutor = Agent(
    name="Math Tutor",
    instructions="""
    You are a patient math tutor.
    Explain the steps clearly and use language appropriate for a student.
    Use the calculate_math tool for arithmetic calculations and explain its result.
    """,
    tools=[math_tool],
)

Create and run the EVS Assistant with its lookup tool:

In [13]:
evs_agent = Agent(
    name="EVS Assistant",
    instructions="""
    You are a helpful Environmental Studies tutor.
    Use the lookup_evs_concept tool for questions about photosynthesis, ozone depletion,
    global warming, and the greenhouse effect. Explain concepts clearly for a student.
    """,
    tools=[evs_lookup_tool],
)

with trace("EVS Assistant with lookup tool"):
    result = await Runner.run(evs_agent, "What is photosynthesis?")
    print(result.final_output)

Photosynthesis is how plants (and some algae and bacteria) make their own food. In simple terms:

- Inputs: sunlight, water (H2O), and carbon dioxide (CO2)
- Process: Uses light energy to convert these into chemical energy
- Output: glucose (a sugar) for the plant to use, and oxygen (O2) released to the air

In equation form (simplified):
6 CO2 + 6 H2O + light energy → C6H12O6 + 6 O2

Key points:
- Takes place mainly in chloroplasts containing chlorophyll (the green pigment)
- Two stages: light-dependent reactions (capture light) and the Calvin cycle (synthesis of glucose)
- Essential for life on Earth: provides the oxygen we breathe and forms the base of most food chains.


In [14]:
with trace("Math Tutor with calculator"):
    result = await Runner.run(
        math_tutor, "What is 7 multiplied by 8? Show me how you got the answer."
    )
    print(result.final_output)

Here's how we get it:

- We’re multiplying 7 by 8.
- One way: think of it as repeated addition: 7 + 7 + 7 + 7 + 7 + 7 + 7 + 7 (eight 7s) = 56.
- Or recall the basic multiplication fact: 7 × 8 = 56.

Answer: 56


Enforce calculator use for arithmetic questions:

In [15]:
math_tutor = Agent(
    name="Math Tutor",
    instructions="""
    You are a patient math tutor.
    Explain the steps clearly and use language appropriate for a student.
    Use the calculate_math tool for arithmetic calculations and explain its result.
    """,
    tools=[math_tool],
    model_settings=ModelSettings(tool_choice="calculate_math"),
)

with trace("Math Tutor with calculator enforced"):
    result = await Runner.run(
        math_tutor, "What is 7 multiplied by 8? Show me how you got the answer."
    )
    print(result.final_output)

Here’s how we get it.

- Multiplication means repeated addition. So 7 × 8 means add 7 to itself 8 times.
- Add 7 eight times: 7 + 7 + 7 + 7 + 7 + 7 + 7 + 7 = 56.

So, 7 multiplied by 8 equals 56.

Quick check (optional): 8 × 7 is the same thing, just add 8 seven times: 8 + 8 + 8 + 8 + 8 + 8 + 8 = 56. The result is consistent: 56.
